# Praktikum Komputasi 3: Simulasi Persamaan Gelombang 1D (Julia)

Persamaan Gelombang 1D memodelkan perambatan sinyal tegangan dan arus pada saluran transmisi tanpa rugi-rugi (*lossless transmission line*), atau vibrasi gelombang mekanik:
$$ \frac{\partial^2 u}{\partial t^2} = v^2 \frac{\partial^2 u}{\partial x^2} $$

dengan:
- $u(x,t)$ adalah amplitudo gelombang (tegangan $v(x,t)$ atau arus $i(x,t)$)
- $v = \frac{1}{\sqrt{LC}}$ adalah kecepatan rambat gelombang pada saluran transmisi ($L$ dan $C$ per satuan panjang)

## Metode Beda Hingga (Finite Difference Method / FDTD)
Diskritisasi turunan parsial kedua waktu dan ruang:
$$ \frac{u_i^{n+1} - 2u_i^n + u_i^{n-1}}{\Delta t^2} = v^2 \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta x^2} $$

Skema pembaruan eksplisit:
$$ u_i^{n+1} = 2(1 - r^2)u_i^n + r^2(u_{i+1}^n + u_{i-1}^n) - u_i^{n-1} $$
dengan $r = \frac{v \Delta t}{\Delta x}$ (Bilangan Courant / CFL condition, syarat stabil $r \le 1$).

In [ ]:
using Plots

## 1. Implementasi Solver Gelombang Beda Hingga 1D

In [ ]:
# Parameter Fisik Saluran Transmisi
L_domain = 10.0     # Panjang saluran (meter)
v = 2.0             # Kecepatan rambat gelombang (m/s)
T_total = 4.0       # Total durasi waktu simulasi (detik)

# Parameter Numerik Grid
Nx = 200            # Jumlah titik spasial
dx = L_domain / (Nx - 1)
x = range(0.0, L_domain, length=Nx)

# Pilih dt sedemikian hingga Courant number r = 0.8 <= 1 (Stabil)
r = 0.8
dt = r * dx / v
Nt = Int(round(T_total / dt))

println("Grid spasi dx = $dx m, langkah waktu dt = $dt s, Nilai Courant r = $r")

## 2. Inisialisasi Pulsa Gelombang dan Perambatan Waktu

In [ ]:
# Matriks solusi: u[spasial, waktu]
u = zeros(Nx, Nt)

# Syarat awal: Pulsa Gauss di tengah saluran (t = 0)
x0 = L_domain / 2.0
sigma = 0.5
u[:, 1] = exp.(-((collect(x) .- x0).^2) ./ (2 * sigma^2))

# Langkah pertama (t = dt) menggunakan syarat kecepatan awal nol (du/dt = 0)
for i in 2:(Nx-1)
    u[i, 2] = u[i, 1] + 0.5 * r^2 * (u[i+1, 1] - 2*u[i, 1] + u[i-1, 1])
end
# Batas Dirichlet (ujung saluran terhubung ke tanah: u(0,t) = 0, u(L,t) = 0)
u[1, 2] = 0.0
u[Nx, 2] = 0.0

# Loop perambatan waktu (FDTD)
for n in 2:(Nt-1)
    for i in 2:(Nx-1)
        u[i, n+1] = 2*(1 - r^2)*u[i, n] + r^2*(u[i+1, n] + u[i-1, n]) - u[i, n-1]
    end
    # Syarat Batas (Refleksi ujung saluran)
    u[1, n+1] = 0.0
    u[Nx, n+1] = 0.0
end

println("Simulasi selesai untuk $Nt langkah waktu.")

## 3. Visualisasi Snapshot Gelombang pada Berbagai Waktu ($t$)

In [ ]:
# Plot snapshot profil gelombang pada beberapa detik
times_to_plot = [0.0, 0.8, 1.6, 2.4, 3.2]

p = plot(xlabel="Posisi Saluran x (meter)", ylabel="Amplitudo Tegangan u(x,t)", 
         title="Perambatan & Pemantulan Gelombang 1D (Julia FDTD)", legend=:topright)

for t_target in times_to_plot
    step_idx = min(Nt, max(1, Int(round(t_target / dt)) + 1))
    t_actual = (step_idx - 1) * dt
    plot!(collect(x), u[:, step_idx], label="t = $(round(t_actual, digits=2)) s", lw=2)
end

display(p)

## 4. Visualisasi Waterfall / Spatiotemporal Surface (2D Heatmap)

In [ ]:
t_axis = range(0.0, T_total, length=Nt)
heatmap(collect(x), collect(t_axis), u', xlabel="Posisi x (m)", ylabel="Waktu t (s)", 
        title="Diagram Perambatan Gelombang Spatiotemporal", color=:viridis)

## 5. Pertanyaan Evaluasi Praktikum
1. **Teorema d'Alembert:** Jelaskan mengapa satu pulsa awal di $t=0$ terbelah menjadi dua pulsa simetris yang merambat ke kiri dan ke kanan!
2. **Refleksi Gelombang:** Amati apa yang terjadi ketika gelombang menabrak batas di $x=0$ dan $x=L$. Mengapa polaritas gelombang terbalik saat dipantulkan (fase bergeser $180^\circ$)?
3. **Kestabilan Courant-Friedrichs-Lewy (CFL):** Ubahlah nilai $r = 1.2$ (melanggar kondisi CFL). Jalankan ulang dan amati ledakan ketidakstabilan numerik (*numerical explosion*)!